In [ ]:
# %% [Cell 1] Imports
import tensorflow as tf
from tensorflow.keras.layers import Conv2D, Conv3D, Flatten, Dense, Reshape, BatchNormalization
from tensorflow.keras.layers import Dropout, Input
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint
from tensorflow.keras.utils import to_categorical
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report, cohen_kappa_score
from operator import truediv
from scipy.io import loadmat
from scipy.signal import savgol_filter
import numpy as np
import matplotlib.pyplot as plt
import scipy.io as sio
import os
from pathlib import Path



In [ ]:
# %% [Cell 2] Load Data
# Updated to use local paths
DATASET_DIR = Path(r"c:/Users/hosse/Desktop/Thesis_Phase1_completed/HSI_WST_Pipeline/dataset")

# Auto-detect Kaggle Environment
if Path('/kaggle/input').exists():
    print("Detected Kaggle Environment. Adjusting paths...")
    found_files = list(Path('/kaggle/input').rglob('Salinas_corrected.mat'))
    if found_files:
        DATASET_DIR = found_files[0].parent
        print(f"Found dataset at: {DATASET_DIR}")
    else:
        print("Warning: Could not find Salinas_corrected.mat in /kaggle/input")

print(f"Loading data from {DATASET_DIR}...")

data_path = DATASET_DIR / 'Salinas_corrected.mat'
gt_path = DATASET_DIR / 'Salinas_gt.mat'

if not data_path.exists():
    raise FileNotFoundError(f"File not found: {data_path}")

# Load into X_raw to preserve original data during interactive runs
X_raw = loadmat(str(data_path))['salinas_corrected']
y_raw = loadmat(str(gt_path))['salinas_gt']

print(f"Data Loaded. Shape: {X_raw.shape}")



In [ ]:
# %% [Cell 3] Constants
test_ratio = 0.8
pixelsize = 25



In [ ]:
# %% [Cell 4] Preprocessing Functions
def padWithZeros(X, margin=2):
    newX = np.zeros((X.shape[0] + 2 * margin, X.shape[1] + 2* margin, X.shape[2]))
    x_offset = margin
    y_offset = margin
    newX[x_offset:X.shape[0] + x_offset, y_offset:X.shape[1] + y_offset, :] = X
    return newX

def select_wavelengths(X, selected_indices):
    """
    Selects specific bands from the hyperspectral cube.
    """
    print(f"Selecting {len(selected_indices)} bands...")
    newX = X[:, :, selected_indices]
    return newX

# --- Advanced Preprocessing (SG, MSC, SNV) ---

def apply_snv(X):
    """ Standard Normal Variate (SNV) """
    print("Applying SNV...")
    # Mean and Std along spectral axis (axis 2)
    mean_vec = np.mean(X, axis=2, keepdims=True)
    std_vec = np.std(X, axis=2, keepdims=True)
    # Avoid division by zero
    return (X - mean_vec) / (std_vec + 1e-8)

def apply_msc(X, reference=None):
    """ Multiplicative Scatter Correction (MSC) """
    print("Applying MSC...")
    original_shape = X.shape
    # Reshape to (N, Bands)
    X_flat = X.reshape(-1, original_shape[2])
    
    if reference is None:
        # Use mean spectrum as reference if not provided
        reference = np.mean(X_flat, axis=0)
    
    n_samples = X_flat.shape[0]
    X_msc = np.zeros_like(X_flat)
    
    # Fit each spectrum to the reference
    for i in range(n_samples):
        # Polyfit returns [slope, intercept]
        fit = np.polyfit(reference, X_flat[i, :], 1, full=True)
        slope = fit[0][0]
        intercept = fit[0][1]
        X_msc[i, :] = (X_flat[i, :] - intercept) / slope
    
    return X_msc.reshape(original_shape)

def apply_sg(X, deriv=0, window_length=30, polyorder=2):
    """ Savitzky-Golay Filter """
    print(f"Applying SG (w={window_length}, p={polyorder}, d={deriv})...")
    return savgol_filter(X, window_length=window_length, polyorder=polyorder, deriv=deriv, axis=2)

def preprocess_pipeline(X, pipeline_name):
    """ 
    Parses pipeline string (e.g., 'SG MSC') and applies transforms in order. 
    Notes:
    - SG/SG1 usually comes first to smooth/differentiate.
    """
    print(f"Preprocessing Pipeline: {pipeline_name}")
    X_proc = X.copy()
    
    # Check for SG variants
    if 'SG' in pipeline_name:
        deriv = 1 if 'SG1' in pipeline_name else 0 
        # Match FCovSel logic: window=30, poly=2
        X_proc = apply_sg(X_proc, deriv=deriv, window_length=30, polyorder=2)
    
    # Check for MSC / SNV
    if 'MSC' in pipeline_name:
        X_proc = apply_msc(X_proc)
    elif 'SNV' in pipeline_name:
        X_proc = apply_snv(X_proc)
        
    return X_proc

def createImageCubes(X, y, windowSize=5, removeZeroLabels=True):
    margin = int((windowSize - 1) / 2)
    zeroPaddedX = padWithZeros(X, margin=margin)
    
    if removeZeroLabels:
        r_indices, c_indices = np.nonzero(y)
        num_samples = len(r_indices)
    else:
        r_indices, c_indices = np.where(y >= 0)
        num_samples = len(r_indices)
        
    print(f"Generating {num_samples} patches (Optimized)...")
    
    patchesData = np.zeros((num_samples, windowSize, windowSize, X.shape[2]), dtype=np.float32)
    patchesLabels = np.zeros((num_samples))
    
    for i in range(num_samples):
        r = r_indices[i]
        c = c_indices[i]
        r_pad = r + margin
        c_pad = c + margin
        patch = zeroPaddedX[r_pad - margin:r_pad + margin + 1, c_pad - margin:c_pad + margin + 1]
        patchesData[i, :, :, :] = patch
        patchesLabels[i] = y[r, c]
        
    if removeZeroLabels:
        patchesLabels -= 1
        
    return patchesData, patchesLabels



In [ ]:
# %% [Cell 5]
print(f"Original X shape: {X_raw.shape}, y shape: {y_raw.shape}")



In [ ]:
# %% [Cell 6] Loop & Experiment Function
import pandas as pd
import ast

def load_selected_bands_from_row(row):
    method_name = row.get('Method', 'Unknown')
    # Get Pipeline from CSV (new column)
    pipeline_name = row.get('Pipeline', 'RAW') 
    if pd.isna(pipeline_name):
        pipeline_name = 'RAW'
    
    # Parse bands
    if 'Selected_Wavelengths' in row:
        bands_str = row['Selected_Wavelengths']
        if isinstance(bands_str, str):
             bands_str = bands_str.replace('[', '').replace(']', '').replace('\n', '')
             selected_wavelengths = [float(x) for x in bands_str.split(',') if x.strip()]
        else:
            selected_wavelengths = list(bands_str)
    elif 'Wavelength' in row:
        selected_wavelengths = [float(x) for x in row['Wavelength']]
    else:
        raise ValueError("Could not find 'Selected_Wavelengths' column")
    
    return selected_wavelengths, method_name, pipeline_name

def get_indices_from_values(selected_values, all_wavelengths):
    indices = []
    for val in selected_values:
        idx = (np.abs(all_wavelengths - val)).argmin()
        indices.append(idx)
    return sorted(list(set(indices)))

# Load Reference Wavelengths
WAVELENGTH_FILE = r"C:\Users\hosse\Desktop\Thesis_Phase1_completed\CCARS_Tuned_Hyperparameters\wavelengths_salinas_corrected_204.csv"

if Path('/kaggle/input').exists():
    found_wl = list(Path('/kaggle/input').rglob('wavelengths_salinas_corrected_204.csv'))
    if found_wl:
        WAVELENGTH_FILE = str(found_wl[0])
        print(f"Found Reference Wavelengths at: {WAVELENGTH_FILE}")

if os.path.exists(WAVELENGTH_FILE):
    all_wl = pd.read_csv(WAVELENGTH_FILE).iloc[:, 0].values
else:
    all_wl = np.arange(204)

# ---------------------------------------------------------
# Define the Experiment Function (Wraps Cells 7-20)
# ---------------------------------------------------------
def train_and_evaluate(method_name, pipeline_name, selected_indices, X_raw, y_raw):
    print(f"\n{'='*50}")
    print(f"Starting Experiment: {method_name} ({pipeline_name})")
    print(f"{'='*50}")

    # 4. Apply Preprocessing based on Pipeline (SG, MSC, etc.)
    if pipeline_name != 'RAW':
        X_processed = preprocess_pipeline(X_raw, pipeline_name)
    else:
        X_processed = X_raw.copy()

    # 5. Select Wavelengths
    X = select_wavelengths(X_processed, selected_indices)
    
    # NOTE: Normalization (StandardScaler) is now applied AFTER split
    # to match FCovSel_SVM_salinas.py perfectly and avoid leakage.
    
    y = y_raw.copy()
    

    # Determine num_classes for spatial split logic
    unique_classes = np.unique(y_raw)
    unique_classes = unique_classes[unique_classes != 0] # Exclude background
    num_classes = len(unique_classes)
    print(f"Detected {num_classes} classes for spatial split.")

    # %% [Cell 8] Block-Based Spatial Train/Test Split (Prevent Data Leakage)
    y_train_map = np.zeros_like(y_raw)
    y_test_map = np.zeros_like(y_raw)
    
    # Split image into 50x50 blocks, assign blocks to Train/Test per class
    H_img, W_img = y_raw.shape
    block_size = 50  
    
    for c in range(1, num_classes + 1):
        class_indices = np.argwhere(y_raw == c)
        if len(class_indices) == 0: continue
            
        # Find unique blocks containing this class
        class_blocks_set = set()
        for r, col in class_indices:
            class_blocks_set.add((r // block_size, col // block_size))
        
        class_blocks = list(class_blocks_set)
        np.random.shuffle(class_blocks)
        
        # Split blocks: 80% Test, 20% Train
        n_test = int(len(class_blocks) * test_ratio)
        if n_test == len(class_blocks) and len(class_blocks) > 1:
            n_test -= 1
            
        test_blocks_set = set(class_blocks[:n_test])
        
        # Assign pixels based on block assignment
        for r, col in class_indices:
            block_id = (r // block_size, col // block_size)
            if block_id in test_blocks_set:
                y_test_map[r, col] = c
            else:
                y_train_map[r, col] = c
    
    # Generate cubes for Train and Test separately
    Xtrain, ytrain = createImageCubes(X, y_train_map, windowSize=pixelsize, removeZeroLabels=True)
    Xtest, ytest = createImageCubes(X, y_test_map, windowSize=pixelsize, removeZeroLabels=True)
    
    print(f"Train shapes: {Xtrain.shape}, Test shapes: {Xtest.shape}")
    
    # %% [Cell 9] Validation Split
    # Split train further into train/valid (random split is fine here as patches are already spatially disjoint from test)
    Xtrain, Xvalid, ytrain, yvalid = train_test_split(Xtrain, ytrain, test_size=0.3333, random_state=42, stratify=ytrain)
    
    # %% [Cell 9b] Apply StandardScaler (Fit on Train, Transform Test/Valid)
    print("Normalizing data (StandardScaler) - Fit on Train...")
    
    # Reshape to (N*H*W, C) for scaling
    # Xtrain shape: (N, 25, 25, Channels)
    N_tr, H, W, C = Xtrain.shape
    Xtrain_reshaped = Xtrain.reshape(-1, C)
    
    scaler = StandardScaler()
    Xtrain_reshaped = scaler.fit_transform(Xtrain_reshaped)
    Xtrain = Xtrain_reshaped.reshape(N_tr, H, W, C)
    
    # Transform Test
    N_te = Xtest.shape[0]
    Xtest = scaler.transform(Xtest.reshape(-1, C)).reshape(N_te, H, W, C)
    
    # Transform Valid
    N_val = Xvalid.shape[0]
    Xvalid = scaler.transform(Xvalid.reshape(-1, C)).reshape(N_val, H, W, C)
    
    print("Normalization Complete.")

    # VERIFICATION: Ensure Model Input has correct number of bands
    current_bands = Xtrain.shape[3]
    expected_bands = len(selected_indices)
    print(f"VERIFICATION: Model Input Shape: {Xtrain.shape}")
    print(f"VERIFICATION: Using {current_bands} bands. Expected: {expected_bands}")
    assert current_bands == expected_bands, f"Error: Model is using {current_bands} bands, but {expected_bands} were selected!"

    # %% [Cell 10] Reshape for CNN
    input_channels = Xtrain.shape[3]
    Xtrain = Xtrain.reshape(-1, pixelsize, pixelsize, input_channels, 1)
    Xtest = Xtest.reshape(-1, pixelsize, pixelsize, input_channels, 1)
    Xvalid = Xvalid.reshape(-1, pixelsize, pixelsize, input_channels, 1)
    
    # %% [Cell 11] One-Hot Encoding
    ytrain = to_categorical(ytrain)
    ytest = to_categorical(ytest)
    yvalid = to_categorical(yvalid)
    num_classes = ytrain.shape[1]

    # %% [Cell 12] Model Definition
    input_shape = (pixelsize, pixelsize, input_channels, 1)

    # %% [Cell 13] Adaptive CNN
    def build_adaptive_cnn(input_shape, num_classes):
        input_layer = Input(input_shape)
        depth = input_shape[2]
        k1 = 7 if depth >= 7 else (5 if depth >= 5 else 3)
        d1 = depth - (k1 - 1)
        k2 = 5 if d1 >= 5 else (3 if d1 >= 3 else 1)
        d2 = d1 - (k2 - 1)
        k3 = 3 if d2 >= 3 else 1
        print(f"Adaptive Kernels: ({k1}), ({k2}), ({k3})")
        
        x = Conv3D(filters=8, kernel_size=(3, 3, k1), activation='relu')(input_layer)
        x = Conv3D(filters=16, kernel_size=(3, 3, k2), activation='relu')(x)
        x = Conv3D(filters=32, kernel_size=(3, 3, k3), activation='relu')(x)
        
        s = x.shape
        x = Reshape((s[1], s[2], s[3] * s[4]))(x)
        x = Conv2D(filters=64, kernel_size=(3,3), activation='relu')(x)
        x = Flatten()(x)
        x = Dense(units=256, activation='relu')(x)
        x = Dropout(0.4)(x)
        x = Dense(units=128, activation='relu')(x)
        x = Dropout(0.4)(x)
        outputs = Dense(units=num_classes, activation='softmax')(x)
        model = Model(inputs=input_layer, outputs=outputs)
        return model

    # %% [Cell 14] Build
    model = build_adaptive_cnn(input_shape, num_classes)
    
    # %% [Cell 15] Compile
    adam = Adam(learning_rate=0.001, decay=1e-06)
    model.compile(loss='categorical_crossentropy', optimizer=adam, metrics=['accuracy'])

    # %% [Cell 16] Checkpoints
    RESULTS_DIR = Path(r"c:/Users/hosse/Desktop/Thesis_Phase1_completed/HSI_Fresh_Adaptation/CNN_Results")
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    
    safe_method_name = f"{method_name}_{pipeline_name}".replace(" ", "_").replace("/", "-")
    model_filename = f"best_model_{safe_method_name}.keras"
    checkpoint_path = RESULTS_DIR / model_filename
    
    checkpoint = ModelCheckpoint(str(checkpoint_path), monitor='val_accuracy', verbose=0, save_best_only=True, mode='max')
    callbacks_list = [checkpoint]

    # %% [Cell 17] Train with Data Augmentation
    from tensorflow.keras.preprocessing.image import ImageDataGenerator

    EPOCHS = 100 
    BATCH_SIZE = 256
    
    # Squeeze the last dimension for ImageDataGenerator: (N, 25, 25, C, 1) -> (N, 25, 25, C)
    Xtrain_aug = Xtrain.squeeze(-1)
    
    # Define Augmentation: Rotations and Flips
    datagen = ImageDataGenerator(
        rotation_range=90,      # Rotate patches
        horizontal_flip=True,   # Flip Left/Right
        vertical_flip=True,     # Flip Up/Down
        fill_mode='reflect'     # Handle borders
    )
    
    # Create Generator
    train_gen = datagen.flow(Xtrain_aug, ytrain, batch_size=BATCH_SIZE)
    
    def expand_dims_generator(generator):
        for x_batch, y_batch in generator:
            # Add back the channel dimension for the 3D CNN input: (B, 25, 25, C) -> (B, 25, 25, C, 1)
            yield np.expand_dims(x_batch, axis=-1), y_batch

    print(f"Training {safe_method_name} with Data Augmentation for {EPOCHS} epochs...")
    
    history = model.fit(
        expand_dims_generator(train_gen),
        steps_per_epoch=len(Xtrain) // BATCH_SIZE,
        validation_data=(Xvalid, yvalid),
        epochs=EPOCHS, 
        callbacks=callbacks_list, 
        verbose=1
    )

    # %% [Cell 18] Evaluate
    model.load_weights(str(checkpoint_path)) # Load best
    model.compile(loss='categorical_crossentropy', optimizer=adam, metrics=['accuracy'])
    loss, accuracy = model.evaluate(Xtest, ytest, batch_size=BATCH_SIZE, verbose=0)
    print(f"Test Accuracy ({safe_method_name}): {accuracy*100:.2f}%")

    # %% [Cell 19] Reports
    Y_pred_test = model.predict(Xtest, verbose=0)
    y_pred_test = np.argmax(Y_pred_test, axis=1)
    y_test_labels = np.argmax(ytest, axis=1)

    classification = classification_report(y_test_labels, y_pred_test)
    confusion = confusion_matrix(y_test_labels, y_pred_test)

    report_path = RESULTS_DIR / f"classification_report_{safe_method_name}.txt"
    with open(report_path, 'w') as f:
        f.write(classification)
        f.write("\n\nConfusion Matrix:\n")
        f.write(str(confusion))
        f.write(f"\n\nTest Accuracy: {accuracy*100:.2f}%")

    # %% [Cell 20] Plotting
    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    plt.plot(history.history['loss'], label='Train Loss')
    plt.plot(history.history['val_loss'], label='Validation Loss')
    plt.title(f'Loss: {safe_method_name}')
    plt.legend(); plt.grid(True)

    plt.subplot(1, 2, 2)
    plt.plot(history.history['accuracy'], label='Train Accuracy')
    plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
    plt.title(f'Accuracy: {safe_method_name}')
    plt.legend(); plt.grid(True)

    plot_path = RESULTS_DIR / f"training_plot_{safe_method_name}.png"
    plt.savefig(plot_path)
    plt.close()
    print(f"Finished {safe_method_name}. Results saved.")


# ---------------------------------------------------------


In [ ]:
# %% [Cell 21] Load Feature CSV
# ---------------------------------------------------------
csv_path = r"c:/Users/hosse/Desktop/Thesis_Phase1_completed/HSI_Fresh_Adaptation/Salinas_all_FCovSel/Salinas_FCovSel_selected_wavelengths.csv"

if Path('/kaggle/input').exists():
    found_csv = list(Path('/kaggle/input').rglob('*selected_wavelengths.csv'))
    if found_csv:
        csv_path = str(found_csv[0])
        print(f"Found Feature CSV at: {csv_path}")
        if len(found_csv) > 1:
            print(f"Warning: Found multiple CSVs: {found_csv}. Using the first one.")

if not os.path.exists(csv_path):
    raise FileNotFoundError(f"File not found: {csv_path}")

df = pd.read_csv(csv_path)
print(f"Loaded {len(df)} methods from CSV.")
print(df[['Method', 'Pipeline', 'N_Selected']].to_string())



In [ ]:
# %% [Cell 22] Experiment 1: FCovSel (RAW)
row = df.iloc[0]
selected_vals, method_name, pipeline_name = load_selected_bands_from_row(row)
selected_indices = get_indices_from_values(selected_vals, all_wl)
train_and_evaluate(method_name, pipeline_name, selected_indices, X_raw, y_raw)



In [ ]:
# %% [Cell 23] Experiment 2: FCovSel (SG MSC)
row = df.iloc[1]
selected_vals, method_name, pipeline_name = load_selected_bands_from_row(row)
selected_indices = get_indices_from_values(selected_vals, all_wl)
train_and_evaluate(method_name, pipeline_name, selected_indices, X_raw, y_raw)



In [ ]:
# %% [Cell 24] Experiment 3: FCovSel (SG SNV)
row = df.iloc[2]
selected_vals, method_name, pipeline_name = load_selected_bands_from_row(row)
selected_indices = get_indices_from_values(selected_vals, all_wl)
train_and_evaluate(method_name, pipeline_name, selected_indices, X_raw, y_raw)



In [ ]:
# %% [Cell 25] Experiment 4: FCovSel (SG1 MSC)
row = df.iloc[3]
selected_vals, method_name, pipeline_name = load_selected_bands_from_row(row)
selected_indices = get_indices_from_values(selected_vals, all_wl)
train_and_evaluate(method_name, pipeline_name, selected_indices, X_raw, y_raw)



In [ ]:
# %% [Cell 26] Experiment 5: FCovSel (SG1 SNV)
row = df.iloc[4]
selected_vals, method_name, pipeline_name = load_selected_bands_from_row(row)
selected_indices = get_indices_from_values(selected_vals, all_wl)
train_and_evaluate(method_name, pipeline_name, selected_indices, X_raw, y_raw)



In [ ]:
# %% [Cell 27] Summary
print("\n" + "="*50)
print("All 5 experiments completed!")
print("="*50)
print("Results saved to CNN_Results/ directory.")

